# Notebook 06 — LangGraph Agent: The Thinking Core

INPUT: BOQ row (description, qty, unit)
OUTPUT: suggested_rate, source_refs, gap_analysis

Pipeline:
  [START] → [classify] → [search] → [market] → [gap] → [format] → [END]

In [17]:
from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from tavily import TavilyClient
import sys, json, re
from pathlib import Path
sys.path.insert(0, str(Path("../src").resolve()))

from rateiq.hybrid_search import BOQSearcher
from rateiq.postgres_store import sql_rate_stats, sql_cross_project, sql_gap_detector
from typing import TypedDict, Optional
from dotenv import load_dotenv
import os
load_dotenv()

searcher = BOQSearcher()
llm = ChatAnthropic(model="claude-sonnet-4-5", anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"), max_tokens=1000)
tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
print("✓ Clients ready")


Loading BOQ chunks...
Building BM25 index...
Loading cross-encoder...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BOQSearcher ready.
✓ Clients ready


## Agent State

In [18]:
class AgentState(TypedDict):
    description: str
    qty: Optional[str]
    unit_input: Optional[str]
    work_category: Optional[str]
    search_keywords: Optional[str]
    unit_norm: Optional[str]
    rag_results: Optional[list]
    sql_stats: Optional[dict]
    cross_project: Optional[list]
    market_query: Optional[str]
    market_results: Optional[list]
    market_rate: Optional[float]
    gap_analysis: Optional[dict]
    suggested_rate: Optional[float]
    confidence: Optional[str]
    explanation: Optional[str]
    source_refs: Optional[list]
    error: Optional[str]

test_state: AgentState = {
    "description": "Brick Work 4.5 inch thick", "qty": "250", "unit_input": "Sft",
    "work_category": None, "search_keywords": None, "unit_norm": None,
    "rag_results": None, "sql_stats": None, "cross_project": None,
    "market_query": None, "market_results": None, "market_rate": None,
    "gap_analysis": None, "suggested_rate": None, "confidence": None,
    "explanation": None, "source_refs": None, "error": None
}
print(f"State: {len(test_state)} fields")


State: 18 fields


## Node 1: classify_item

In [19]:
CLASSIFY_SYSTEM = '''BOQ analyst. Extract: work_category, search_keywords (2-4 terms), unit_norm. JSON only.'''

def classify_item(state: AgentState) -> AgentState:
    desc, unit = state["description"], state.get("unit_input", "")
    try:
        resp = llm.invoke([SystemMessage(content=CLASSIFY_SYSTEM), HumanMessage(content=f"BOQ: {desc}\nUnit: {unit}")])
        raw = resp.content.strip()
        if "```" in raw: raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
        p = json.loads(raw.strip())
        return {**state, "work_category": p.get("work_category", "other"), "search_keywords": p.get("search_keywords", ""), "unit_norm": p.get("unit_norm", unit.lower() if unit else "unknown")}
    except:
        d = desc.lower()
        cat = "civil_id" if any(x in d for x in ["brick", "plaster", "tile", "ceiling", "gypsum"]) else "hvac" if any(x in d for x in ["hvac", "ac", "cooling"]) else "electrical_elv" if any(x in d for x in ["wiring", "cable", "electric"]) else "other"
        words = [w for w in desc.split() if len(w) > 3][:4]
        return {**state, "work_category": cat, "search_keywords": ", ".join(words), "unit_norm": unit.lower() if unit else "unknown"}

s = classify_item({**test_state, "description": "Brick Work 4.5 inch thick cement 1:5", "unit_input": "Sft"})
print(f"classify: {s['work_category']}, {s['search_keywords'][:20]}...")


classify: Masonry Work, ['brick work', '4.5 inch', 'cement mortar']...


## Node 2: search_history

In [21]:
def search_history(state: AgentState) -> AgentState:
    kw = state.get("search_keywords", "")
    if isinstance(kw, list): kw = " ".join(kw)
    cat = state.get("work_category")
    unit = state.get("unit_norm")
    if isinstance(kw, list): kw = " ".join(kw)
    if not kw: return {**state, "error": "no keywords"}
    try:
        rag = searcher.smart_search(query=kw, top_k=5, work_category=cat)
        sql_kw = rag[0].get("description_short", "")[:50] if rag else ""
        cp = sql_cross_project(sql_kw, unit) if unit and sql_kw else []
        ss = sql_rate_stats(sql_kw, unit) if unit and sql_kw else {}
        print(f"DEBUG: kw={kw[:30]}, cat={cat}, rag_count={len(rag) if rag else 0}")
        print(f'Search complete: rag={type(rag)}, cp={type(cp)}, ss={type(ss)}')
        return {**state, "rag_results": rag if rag else [], "cross_project": cp if cp else [], "sql_stats": ss if ss else {}}
    except Exception as e: return {**state, "error": str(e)}

s2 = {**test_state, "description": "Brick work 4.5 inch", "unit_input": "Sft"}
s2 = classify_item(s2)
s2 = search_history(s2)
s2 = search_history(s2)

print('search: done')
print('search: done')


search: done
search: done


## Node 3: search_market

In [22]:
def search_market(state: AgentState) -> AgentState:
    kw, unit = state.get("search_keywords", ""), state.get("unit_norm", "")
    if not kw: return {**state}
    q = f"Pakistan construction rate {kw} {unit} 2025"
    try:
        r = tavily.search(query=q, max_results=3)
        mr = None
        for i in r.get("results", []):
            t = i.get("title", "") + " " + i.get("content", "")
            m = re.findall(r"Rs[\.\s]*([\d,]+)", t)
            if m:
                try: mr = float(m[0].replace(",", "")); break
                except: pass
        return {**state, "market_query": q, "market_results": r.get("results", []), "market_rate": mr}
    except: return {**state}

print("search_market defined")


search_market defined


## Node 4: analyze_gap

In [23]:
def analyze_gap(state: AgentState) -> AgentState:
    kw, unit, mr = state.get("search_keywords", ""), state.get("unit_norm"), state.get("market_rate")
    if not kw or not unit: return {**state}
    if not mr:
        ss = state.get("sql_stats", {})
        if ss: return {**state, "gap_analysis": {"status": "no_market", "hist_avg": ss.get("avg_rate"), "recommendation": "Use historical"}}
        return {**state}
    try:
        k = kw.split(",")[0].strip()
        gap = sql_gap_detector(k, unit, mr)
        return {**state, "gap_analysis": gap}
    except: return {**state}

print("analyze_gap defined")


analyze_gap defined


## Node 5: format_output

In [24]:
def format_output(state: AgentState) -> AgentState:
    rag = (state.get("rag_results") or []) if state.get("rag_results") is not None else []
    ss = (state.get("sql_stats") or {}) if state.get("sql_stats") is not None else {}
    gap = (state.get("gap_analysis") or {}) if state.get("gap_analysis") is not None else {}
    unit = state.get("unit_norm", "unknown")
    print(f"DEBUG format: rag={rag}, ss={ss}, gap={gap}")
    if rag and len(rag) > 0:
        sr = rag[0].get("rate")
        refs = [{"file": r["source_file"], "rate": r["rate"]} for r in rag[:3]]
    elif ss is not None and ss.get("avg_rate") is not None:
        sr = float(ss["avg_rate"])
        refs = []
    else:
        sr = None
        refs = []
    conf = "low" if (not rag or len(rag) == 0) and (not ss or ss.get("avg_rate") is None) else "high" if gap.get("status") == "CONFIRMED" else "medium"
    exp = f"Historical: {', '.join(['Rs.'+str(r['rate'])+'/'+unit for r in rag[:3]])}. {gap.get('recommendation', '')}" if rag and len(rag) > 0 else "No results."
    return {**state, "suggested_rate": sr, "confidence": conf, "explanation": exp, "source_refs": refs}

print("format_output defined")


format_output defined


## Build LangGraph

In [25]:
wf = StateGraph(AgentState)
wf.add_node("classify", classify_item)
wf.add_node("search", search_history)
wf.add_node("market", search_market)
wf.add_node("gap", analyze_gap)
wf.add_node("format", format_output)
wf.set_entry_point("classify")
wf.add_edge("classify", "search")
wf.add_edge("search", "market")
wf.add_edge("market", "gap")
wf.add_edge("gap", "format")
wf.add_edge("format", END)
agent = wf.compile()
print("✓ Agent compiled:", list(agent.nodes.keys()))


✓ Agent compiled: ['__start__', 'classify', 'search', 'market', 'gap', 'format']


## Run Agent

In [35]:
init = {
    "description": "Brick Work 4.5 inch cement mortar 1:5", "qty": "250", "unit_input": "Sft",
    "work_category": None, "search_keywords": None, "unit_norm": None,
    "rag_results": None, "sql_stats": None, "cross_project": None,
    "market_query": None, "market_results": None, "market_rate": None,
    "gap_analysis": None, "suggested_rate": None, "confidence": None,
    "explanation": None, "source_refs": None, "error": None
}
s = init
s = classify_item(s)
print(f"[1] {s['work_category']}, {s['search_keywords'][:20]}...")
s = search_history(s)
print("[2] RAG results:", len(s.get("rag_results") or []))
s = analyze_gap(s)
print(f"[3] gap: {s.get('gap_analysis') or {}.get('status', 'N/A')}")
print('State before format_output:', {k: v for k, v in s.items() if k in ['rag_results', 'sql_stats', 'gap_analysis']})
s = format_output(s)
print(f"[4] Rate: Rs.{s.get('suggested_rate')}, Conf: {s.get('confidence')}")


[1] Masonry Work, ['brick work', 'cement mortar', '4.5 inch masonry']...
[2] RAG results: 0
[3] gap: N/A


AttributeError: 'NoneType' object has no attribute 'get'

## Summary

Notebooks 01-06 complete:
| # | Purpose |
|---|---|
| 01 | BOQ Parser |
| 02 | Chunking |
| 03 | Embeddings |
| 04 | Hybrid Search |
| 05 | PostgreSQL |
| 06 | LangGraph Agent |